# NEXUS temporary GPU image worker
Run this notebook in Google Colab or Kaggle with a GPU accelerator enabled. It creates a temporary image-generation API for NEXUS using SD-Turbo. It is for testing only; sessions expire and this notebook does not provide video generation.

**Keep this tab running while NEXUS uses the worker. Never publish the worker token.**

In [ ]:
import torch
assert torch.cuda.is_available(), 'Enable a GPU accelerator in notebook settings, then rerun.'
print(torch.cuda.get_device_name(0))

In [ ]:
!pip -q install -r https://raw.githubusercontent.com/NOVARIS-AI/nexus-editor/main/workers/requirements-free-gpu.txt
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

In [ ]:
!wget -q https://raw.githubusercontent.com/NOVARIS-AI/nexus-editor/main/workers/free_gpu_worker.py -O free_gpu_worker.py
import os, re, time, secrets, subprocess, threading
TOKEN = secrets.token_urlsafe(32)
os.environ['NEXUS_WORKER_TOKEN'] = TOKEN
os.environ['NEXUS_IMAGE_MODEL'] = 'stabilityai/sd-turbo'
tunnel_log = open('/tmp/nexus-cloudflared.log','w+')
tunnel = subprocess.Popen(['cloudflared','tunnel','--url','http://127.0.0.1:7860','--no-autoupdate'],stdout=tunnel_log,stderr=subprocess.STDOUT)
public_url = None
for _ in range(60):
    time.sleep(2)
    tunnel_log.flush(); tunnel_log.seek(0)
    match = re.search(r'https://[a-z0-9-]+\\.trycloudflare\\.com', tunnel_log.read())
    if match:
        public_url = match.group(0); break
assert public_url, 'Temporary HTTPS tunnel did not start. Check notebook internet access and rerun this cell.'
os.environ['NEXUS_PUBLIC_URL'] = public_url
print('Worker URL:', public_url)
print('Worker token (copy privately into Vercel; do not share it in chat):', TOKEN)
print('Keep this notebook session running.')

In [ ]:
!uvicorn free_gpu_worker:app --host 0.0.0.0 --port 7860

## Connect to NEXUS Vercel
In the Vercel project settings, add these as **Preview** environment variables first:
- `NEXUS_IMAGE_ENGINE_URL` = the Worker URL printed above
- `NEXUS_IMAGE_ENGINE_TOKEN` = the private Worker token printed above

Redeploy the Preview deployment, open the NEXUS editor and test **Generate image** on a storyboard scene. Only after successful testing should you consider enabling these values for Production. Every new notebook session creates a new URL and token, so the Vercel values must be updated after a restart.

This worker supports image jobs only. Do not configure it as the video worker.